# Mappe interattive delle preferenze PD a Roma

Questo notebook genera tre mappe HTML in `docs/maps/`.

- `roma_pd_comune_2021.html`: Roma, elezione comunale 2021.
- `municipio3_pd_comune_2021.html`: focus sul Municipio III per l'elezione comunale.
- `municipio3_pd_municipio_2021.html`: focus sul Municipio III per l'elezione municipale.

La coropleta usa la percentuale del PD sul totale dei voti di lista per sezione. Nei popup compaiono i voti alla lista PD e la top 10 delle preferenze ai consiglieri, con etichette in italiano.

Sui quartieri viene aggiunto un layer testuale dinamico: a zoom basso compare solo il cognome del candidato piu forte, poi con lo zoom compaiono anche gli altri fino ai primi 5.

In [28]:
from pathlib import Path
import html

import folium
import geopandas as gpd
import pandas as pd
from folium.features import DivIcon

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = ROOT / "dati_elezioni"
GEO_DIR = ROOT / "geoframes"
DOCS_DIR = ROOT / "docs"
MAPS_DIR = DOCS_DIR / "maps"
DATA_OUT_DIR = DOCS_DIR / "data"

MAPS_DIR.mkdir(parents=True, exist_ok=True)
DATA_OUT_DIR.mkdir(parents=True, exist_ok=True)

PD_2021 = "PD PARTITO DEMOCRATICO"
ROOT

WindowsPath('C:/Users/gabri/Documents/GitHub/preferenze_roma')

In [29]:
def normalize_sections(df: pd.DataFrame, column: str = "SEZIONE") -> pd.DataFrame:
    out = df.copy()
    out[column] = pd.to_numeric(out[column], errors="coerce")
    out = out.dropna(subset=[column])
    out[column] = out[column].astype(int)
    return out


def clean_match_quartieri(df: pd.DataFrame) -> pd.DataFrame:
    cols = [c for c in df.columns if not str(c).startswith("Unnamed") and c != "H1"]
    out = df[cols].copy()
    out = normalize_sections(out)
    out["COD_ASC"] = out["COD_ASC"].astype(str)
    out["NOME_QUARTIERE"] = out["NOME_QUARTIERE"].astype(str)
    return out[["SEZIONE", "COD_ASC", "NOME_QUARTIERE"]].drop_duplicates()


def surname_only(full_name: str) -> str:
    if not isinstance(full_name, str):
        return ""
    parts = [p for p in full_name.strip().split() if p]
    return parts[-1] if parts else full_name


def build_top10_per_section(preferences: pd.DataFrame, list_name: str, output_col: str) -> pd.DataFrame:
    filtered = preferences.loc[preferences["LISTA"] == list_name].copy()
    filtered["PREFERENZE"] = pd.to_numeric(filtered["PREFERENZE"], errors="coerce").fillna(0)
    grouped = (
        filtered.groupby(["SEZIONE", "CANDIDATO"], as_index=False)["PREFERENZE"]
        .sum()
        .sort_values(["SEZIONE", "PREFERENZE", "CANDIDATO"], ascending=[True, False, True])
    )
    top10 = grouped.groupby("SEZIONE").head(10).copy()
    top10["rank"] = top10.groupby("SEZIONE").cumcount() + 1
    top10["riga"] = (
        top10["rank"].astype(str)
        + ". "
        + top10["CANDIDATO"]
        + " ("
        + top10["PREFERENZE"].round(0).astype(int).astype(str)
        + ")"
    )
    return top10.groupby("SEZIONE")["riga"].apply(" | ".join).reset_index(name=output_col)


def build_popup_frame(df: pd.DataFrame, top_col: str, tipo_label: str) -> pd.DataFrame:
    out = df.copy()
    out["Percentuale PD"] = (out["pd_share"].fillna(0) * 100).round(2).astype(str) + "%"
    out["Voti lista PD"] = out["pd_votes"].fillna(0).round(0).astype(int)
    out["Affluenza"] = (out["affluenza"].fillna(0) * 100).round(2).astype(str) + "%"
    out["Classifica top 10 preferenze"] = out[top_col].fillna("Dato non disponibile")
    out["Tipo elezione"] = tipo_label
    return out


def add_title_box(map_obj: folium.Map, title: str, subtitle: str) -> None:
    box = f"""
    <div style='position: fixed; z-index: 9999; left: 16px; top: 16px; max-width: 360px; background: rgba(255,255,255,0.95); border-radius: 12px; padding: 14px 16px; box-shadow: 0 10px 28px rgba(0,0,0,0.16); font-family: Arial, sans-serif;'>
      <div style='font-size: 18px; font-weight: 700; color: #183a1d; margin-bottom: 6px;'>{html.escape(title)}</div>
      <div style='font-size: 13px; line-height: 1.45; color: #355e3b;'>{html.escape(subtitle)}</div>
    </div>
    """
    map_obj.get_root().html.add_child(folium.Element(box))


def compute_quartiere_strength(preferences: pd.DataFrame, match_quartieri: pd.DataFrame, quartieri_geom: gpd.GeoDataFrame, list_name: str) -> gpd.GeoDataFrame:
    filtered = preferences.loc[preferences["LISTA"] == list_name].copy()
    filtered["PREFERENZE"] = pd.to_numeric(filtered["PREFERENZE"], errors="coerce").fillna(0)
    merged = filtered.merge(match_quartieri, on="SEZIONE", how="inner")
    by_quartiere_candidate = (
        merged.groupby(["COD_ASC", "NOME_QUARTIERE", "CANDIDATO"], as_index=False)["PREFERENZE"]
        .sum()
        .sort_values(["COD_ASC", "PREFERENZE", "CANDIDATO"], ascending=[True, False, True])
    )
    top5 = by_quartiere_candidate.groupby("COD_ASC").head(5).copy()
    top5["label_name"] = top5["CANDIDATO"].apply(surname_only) + " (" + top5["PREFERENZE"].round(0).astype(int).astype(str) + ")"
    pairs = top5.groupby("COD_ASC").apply(lambda grp: list(zip(grp["label_name"], grp["PREFERENZE"]))).reset_index(name="top_candidates_quartiere")
    quartieri = quartieri_geom.merge(pairs, on="COD_ASC", how="left")
    quartieri["top_candidates_quartiere"] = quartieri["top_candidates_quartiere"].apply(lambda x: x if isinstance(x, list) else [])
    quartieri["label_point"] = quartieri.representative_point()
    return quartieri


def add_quartieri_zoom_labels(map_obj: folium.Map, quartieri_gdf: gpd.GeoDataFrame, layer_name: str = "Consiglieri forti per quartiere") -> None:
    layer = folium.FeatureGroup(name=layer_name, show=True)
    for _, row in quartieri_gdf.iterrows():
        entries = row.get("top_candidates_quartiere", [])
        if not entries:
            continue

        point = row["label_point"]
        quartiere = html.escape(str(row["DEN_Z_URB"]))
        spans = []
        for idx, (label, votes) in enumerate(entries, start=1):
            if votes <= 0:
                continue
            size = 16 if idx == 1 else (14 if idx == 2 else (12 if idx == 3 else (11 if idx == 4 else 10)))
            css_class = f"quartiere-word quartiere-word-{idx}"
            spans.append(
                f"<span class='{css_class}' style='font-size:{size}px; font-weight:700; color:#0f5132; opacity:0.56; display:inline-block; margin:0 4px 2px;'>"
                f"{html.escape(label)}</span>"
            )

        if not spans:
            continue

        block = f"""
        <div class='quartiere-label-box' style='width:240px; text-align:center; transform: translate(-120px, -8px); font-family: Arial, sans-serif;'>
          <div style='font-size:10px; letter-spacing:0.08em; text-transform:uppercase; color:#14532d; opacity:0.50; margin-bottom:3px;'>{quartiere}</div>
          <div>{''.join(spans)}</div>
        </div>
        """
        folium.Marker(location=[point.y, point.x], icon=DivIcon(html=block)).add_to(layer)

    layer.add_to(map_obj)

    map_name = map_obj.get_name()
    js = f"""
    <script>
    function updateQuartiereWords_{map_name}() {{
      var zoom = {map_name}.getZoom();
      document.querySelectorAll('.quartiere-word-1').forEach(function(el) {{ el.style.display = ''; }});
      document.querySelectorAll('.quartiere-word-2').forEach(function(el) {{ el.style.display = zoom >= 11 ? '' : 'none'; }});
      document.querySelectorAll('.quartiere-word-3').forEach(function(el) {{ el.style.display = zoom >= 12 ? '' : 'none'; }});
      document.querySelectorAll('.quartiere-word-4').forEach(function(el) {{ el.style.display = zoom >= 13 ? '' : 'none'; }});
      document.querySelectorAll('.quartiere-word-5').forEach(function(el) {{ el.style.display = zoom >= 14 ? '' : 'none'; }});
    }}
    {map_name}.whenReady(function() {{
      updateQuartiereWords_{map_name}();
      {map_name}.on('zoomend', updateQuartiereWords_{map_name});
    }});
    </script>
    """
    map_obj.get_root().html.add_child(folium.Element(js))


def save_choropleth_map(
    gdf: gpd.GeoDataFrame,
    quartieri_layer: gpd.GeoDataFrame,
    output_name: str,
    title: str,
    subtitle: str,
    popup_tipo_label: str,
    top_col: str,
    cmap: str = "Greens",
):
    popup_gdf = build_popup_frame(gdf, top_col=top_col, tipo_label=popup_tipo_label)
    m = popup_gdf.explore(
        column="pd_share",
        cmap=cmap,
        tiles="CartoDB positron",
        tooltip=["SEZIONE", "Percentuale PD", "Voti lista PD"],
        popup=["SEZIONE", "Tipo elezione", "Percentuale PD", "Voti lista PD", "Affluenza", "Classifica top 10 preferenze"],
        popup_kwds={
            "aliases": ["Sezione", "Tipo elezione", "Percentuale PD", "Voti lista PD", "Affluenza", "Classifica top 10 preferenze"],
            "labels": True,
        },
        legend=True,
        style_kwds={"weight": 0.08, "color": "#1f2937", "fillOpacity": 0.76},
        highlight_kwds={"weight": 0.28, "color": "#14532d", "fillOpacity": 0.86},
        name=title,
    )
    add_quartieri_zoom_labels(m, quartieri_layer)
    add_title_box(m, title, subtitle)
    folium.LayerControl(collapsed=False).add_to(m)
    output_path = MAPS_DIR / output_name
    m.save(output_path)
    return output_path


In [30]:
sezioni = normalize_sections(gpd.read_file(GEO_DIR / "precincts_roma_bulding.geojson"))[["SEZIONE", "geometry"]]
quartieri_geom = gpd.read_file(GEO_DIR / "roma_quartieri.geojson")[["COD_ASC", "DEN_Z_URB", "geometry"]].copy()
match_quartieri = clean_match_quartieri(pd.read_csv(GEO_DIR / "roma_match_quartieri.csv"))

liste_comune_2021 = normalize_sections(pd.read_csv(DATA_DIR / "comunali_sindaco_lista_2021.csv"))
preferenze_comune_2021 = normalize_sections(pd.read_csv(DATA_DIR / "comunali_preferenze_2021.csv"))

liste_municipio3_2021 = normalize_sections(pd.read_csv(DATA_DIR / "comunali_liste_municipio_3_2021.csv"))
preferenze_municipio3_2021 = normalize_sections(pd.read_csv(DATA_DIR / "comunali_preferenze_2021_municipio_3_2021.csv"))

sezioni.head()

,SEZIONE,geometry
0,1,"MULTIPOLYGON (((12.54817 41.95611, 12.54826 41.95607, 12.54836 41.95621, 12.54856 41.95613, 12.548 41.95534, 12.5478 41.95542, 12.54789 41.95554, 12.54779 41.95558, 12.5477 41.95546, 12.54754 41.9..."
1,2,"MULTIPOLYGON (((12.50049 41.89794, 12.50043 41.89789, 12.50028 41.89797, 12.50004 41.89811, 12.50002 41.89814, 12.50004 41.89816, 12.50027 41.8984, 12.5005 41.89863, 12.50067 41.89853, 12.50107 41..."
2,3,"MULTIPOLYGON (((12.50328 41.89614, 12.50301 41.89631, 12.50263 41.89654, 12.50242 41.89666, 12.50257 41.89681, 12.50279 41.89704, 12.50293 41.89717, 12.50313 41.89705, 12.50352 41.89681, 12.50372 ..."
3,4,"MULTIPOLYGON (((12.50033 41.89515, 12.5002 41.89538, 12.50008 41.8956, 12.50001 41.89572, 12.50013 41.89572, 12.50018 41.89565, 12.50019 41.89563, 12.50024 41.89565, 12.50028 41.89558, 12.50045 41..."
4,5,"MULTIPOLYGON (((12.50078 41.89434, 12.50068 41.8945, 12.50059 41.89468, 12.50047 41.89488, 12.5005 41.89491, 12.50068 41.8949, 12.50088 41.89488, 12.50113 41.89486, 12.50144 41.89484, 12.50169 41...."


In [31]:
top10_comune = build_top10_per_section(preferenze_comune_2021, PD_2021, "top10_preferenze")
top10_municipio = build_top10_per_section(preferenze_municipio3_2021, PD_2021, "top10_preferenze")

mappa_comune = liste_comune_2021[["SEZIONE", "TOTALE", PD_2021, "AFFLUENZA"]].copy()
mappa_comune = mappa_comune.rename(columns={"TOTALE": "totale_lista", PD_2021: "pd_votes", "AFFLUENZA": "affluenza"})
mappa_comune["pd_share"] = mappa_comune["pd_votes"] / mappa_comune["totale_lista"]
mappa_comune = sezioni.merge(mappa_comune, on="SEZIONE", how="inner").merge(top10_comune, on="SEZIONE", how="left")

mun3_sections = set(liste_municipio3_2021["SEZIONE"].unique())
mappa_comune_mun3 = mappa_comune.loc[mappa_comune["SEZIONE"].isin(mun3_sections)].copy()

mappa_municipio = liste_municipio3_2021[["SEZIONE", "TOTALE", PD_2021, "AFFLUENZA"]].copy()
mappa_municipio = mappa_municipio.rename(columns={"TOTALE": "totale_lista", PD_2021: "pd_votes", "AFFLUENZA": "affluenza"})
mappa_municipio["pd_share"] = mappa_municipio["pd_votes"] / mappa_municipio["totale_lista"]
mappa_municipio = sezioni.merge(mappa_municipio, on="SEZIONE", how="inner").merge(top10_municipio, on="SEZIONE", how="left")

mappa_comune.head()

,SEZIONE,geometry,totale_lista,pd_votes,affluenza,pd_share,top10_preferenze
0,1,"MULTIPOLYGON (((12.54817 41.95611, 12.54826 41.95607, 12.54836 41.95621, 12.54856 41.95613, 12.548 41.95534, 12.5478 41.95542, 12.54789 41.95554, 12.54779 41.95558, 12.5477 41.95546, 12.54754 41.9...",525,83,0.396526,0.158095,1. CARLA CONSUELO FERMARIELLO (7) | 2. YURI TROMBETTI (5) | 3. ANDREA ALEMANNI (4) | 4. MAURIZIO VELOCCIA (4) | 5. DANIELE PARRUCCI (3) | 6. GIULIA TEMPESTA (2) | 7. RICCARDO CORBUCCI (2) | 8. SVE...
1,2,"MULTIPOLYGON (((12.50049 41.89794, 12.50043 41.89789, 12.50028 41.89797, 12.50004 41.89811, 12.50002 41.89814, 12.50004 41.89816, 12.50027 41.8984, 12.5005 41.89863, 12.50067 41.89853, 12.50107 41...",289,53,0.456556,0.183391,1. SABRINA ALFONSI (5) | 2. GIULIA TEMPESTA (4) | 3. MAURIZIO VELOCCIA (4) | 4. MASSIMO DE SIMONI (2) | 5. YURI TROMBETTI (2) | 6. ANTONIO STAMPETE (1) | 7. DARIO MARCUCCI (1) | 8. NELLA CONVERTI ...
2,3,"MULTIPOLYGON (((12.50328 41.89614, 12.50301 41.89631, 12.50263 41.89654, 12.50242 41.89666, 12.50257 41.89681, 12.50279 41.89704, 12.50293 41.89717, 12.50313 41.89705, 12.50352 41.89681, 12.50372 ...",333,55,0.417817,0.165165,1. YIFAN LIN (9) | 2. ANDREA ALEMANNI (3) | 3. GIAMMARCO PALMIERI (3) | 4. MAURIZIO VELOCCIA (3) | 5. SABRINA ALFONSI (3) | 6. MARIANO ANGELUCCI (2) | 7. NELLA CONVERTI (2) | 8. SVETLANA CELLI (2)...
3,4,"MULTIPOLYGON (((12.50033 41.89515, 12.5002 41.89538, 12.50008 41.8956, 12.50001 41.89572, 12.50013 41.89572, 12.50018 41.89565, 12.50019 41.89563, 12.50024 41.89565, 12.50028 41.89558, 12.50045 41...",261,34,0.243017,0.130268,1. MAURIZIO VELOCCIA (4) | 2. SABRINA ALFONSI (4) | 3. CARLA CONSUELO FERMARIELLO (2) | 4. GIOVANNI CRISANTI (1) | 5. NELLA CONVERTI (1) | 6. STEFANO MARONGIU (1) | 7. ALESSANDRO LEPIDINI (0) | 8....
4,5,"MULTIPOLYGON (((12.50078 41.89434, 12.50068 41.8945, 12.50059 41.89468, 12.50047 41.89488, 12.5005 41.89491, 12.50068 41.8949, 12.50088 41.89488, 12.50113 41.89486, 12.50144 41.89484, 12.50169 41....",371,67,0.416386,0.180593,1. GIORGIO RAFFAELE CALVI (10) | 2. SABRINA ALFONSI (10) | 3. MAURIZIO VELOCCIA (4) | 4. YURI TROMBETTI (3) | 5. ANTONELLA MELITO (2) | 6. ANTONIO CIANCIO (2) | 7. ANTONIO STAMPETE (2) | 8. FLAVIA...


In [32]:
quartieri_comune = compute_quartiere_strength(preferenze_comune_2021, match_quartieri, quartieri_geom, PD_2021)
codici_mun3 = match_quartieri.loc[match_quartieri["SEZIONE"].isin(mun3_sections), "COD_ASC"].unique()
quartieri_mun3_comune = quartieri_comune.loc[quartieri_comune["COD_ASC"].isin(codici_mun3)].copy()
quartieri_mun3_municipio = compute_quartiere_strength(preferenze_municipio3_2021, match_quartieri.loc[match_quartieri["SEZIONE"].isin(mun3_sections)], quartieri_geom, PD_2021)
quartieri_mun3_municipio = quartieri_mun3_municipio.loc[quartieri_mun3_municipio["COD_ASC"].isin(codici_mun3)].copy()

quartieri_comune[["COD_ASC", "DEN_Z_URB", "top_candidates_quartiere"]].head()

C:\Users\gabri\AppData\Local\Temp\ipykernel_42956\2997158894.py:77: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pairs = top5.groupby("COD_ASC").apply(lambda grp: list(zip(grp["label_name"], grp["PREFERENZE"]))).reset_index(name="top_candidates_quartiere")
C:\Users\gabri\AppData\Local\Temp\ipykernel_42956\2997158894.py:77: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pairs = top5.groupby("COD_ASC").apply(

,COD_ASC,DEN_Z_URB,top_candidates_quartiere
0,10l,Morena,"[(BATTAGLIA (118), 118.0), (STELITANO (52), 52.0), (ALEMANNI (49), 49.0), (CELLI (49), 49.0), (STAMPETE (48), 48.0)]"
1,10a,Don Bosco,"[(PARRUCCI (128), 128.0), (SIMONI (115), 115.0), (STELITANO (114), 114.0), (BATTAGLIA (105), 105.0), (CONVERTI (93), 93.0)]"
2,10b,Appio-Claudio,"[(SIMONI (129), 129.0), (STELITANO (123), 123.0), (BATTAGLIA (103), 103.0), (CONVERTI (91), 91.0), (PARRUCCI (87), 87.0)]"
3,10c,Quarto Miglio,"[(STELITANO (152), 152.0), (BATTAGLIA (114), 114.0), (STAMPETE (36), 36.0), (PARRUCCI (31), 31.0), (TEMPESTA (18), 18.0)]"
4,10d,Pignatelli,"[(PARRUCCI (233), 233.0), (BATTAGLIA (16), 16.0), (STELITANO (15), 15.0), (PAPPATA' (13), 13.0), (CELLI (9), 9.0)]"


In [33]:
out_roma = save_choropleth_map(
    gdf=mappa_comune,
    quartieri_layer=quartieri_comune,
    output_name="roma_pd_comune_2021.html",
    title="Roma 2021: percentuale PD e preferenze dei consiglieri comunali",
    subtitle="La coropleta usa la percentuale del PD sul totale dei voti di lista. Nei popup sono riportati i voti alla lista e la top 10 delle preferenze dei consiglieri comunali PD.",
    popup_tipo_label="Elezione comunale 2021",
    top_col="top10_preferenze",
    cmap="BuGn",
)

out_roma

WindowsPath('C:/Users/gabri/Documents/GitHub/preferenze_roma/docs/maps/roma_pd_comune_2021.html')

In [34]:
out_mun3_comune = save_choropleth_map(
    gdf=mappa_comune_mun3,
    quartieri_layer=quartieri_mun3_comune,
    output_name="municipio3_pd_comune_2021.html",
    title="Municipio III 2021: percentuale PD e preferenze dei consiglieri comunali",
    subtitle="Focus sul Municipio III per l'elezione comunale. A zoom basso si vede il cognome del consigliere piu forte per quartiere, poi compaiono progressivamente anche gli altri fino ai primi 5.",
    popup_tipo_label="Elezione comunale 2021",
    top_col="top10_preferenze",
    cmap="Greens",
)

out_mun3_comune

WindowsPath('C:/Users/gabri/Documents/GitHub/preferenze_roma/docs/maps/municipio3_pd_comune_2021.html')

In [35]:
out_mun3_municipio = save_choropleth_map(
    gdf=mappa_municipio,
    quartieri_layer=quartieri_mun3_municipio,
    output_name="municipio3_pd_municipio_2021.html",
    title="Municipio III 2021: percentuale PD e preferenze dei consiglieri municipali",
    subtitle="La coropleta usa la percentuale del PD nelle elezioni municipali. Nei popup compare la top 10 delle preferenze dei consiglieri municipali PD.",
    popup_tipo_label="Elezione municipale 2021",
    top_col="top10_preferenze",
    cmap="Reds",
)

out_mun3_municipio

WindowsPath('C:/Users/gabri/Documents/GitHub/preferenze_roma/docs/maps/municipio3_pd_municipio_2021.html')

In [36]:
summary_comune = mappa_comune[["SEZIONE", "pd_votes", "pd_share", "top10_preferenze"]].copy()
summary_comune["tipo"] = "comune_2021"
summary_municipio = mappa_municipio[["SEZIONE", "pd_votes", "pd_share", "top10_preferenze"]].copy()
summary_municipio["tipo"] = "municipio3_2021"

summary_comune.to_csv(DATA_OUT_DIR / "pd_section_summary_comune_2021.csv", index=False)
summary_municipio.to_csv(DATA_OUT_DIR / "pd_section_summary_municipio3_2021.csv", index=False)

list(MAPS_DIR.glob("*.html"))

[WindowsPath('C:/Users/gabri/Documents/GitHub/preferenze_roma/docs/maps/municipio3_pd_comune_2021.html'),
 WindowsPath('C:/Users/gabri/Documents/GitHub/preferenze_roma/docs/maps/municipio3_pd_municipio_2021.html'),
 WindowsPath('C:/Users/gabri/Documents/GitHub/preferenze_roma/docs/maps/roma_pd_comune_2021.html')]